In [6]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000

income = np.random.normal(60000, 15000, n)
debt = np.random.normal(20000, 10000, n)
credit_score = np.random.normal(650, 70, n)
loan_amount = np.random.normal(25000, 12000, n)
employment_years = np.random.uniform(0, 15, n)
age = np.random.uniform(21, 65, n)

# Risk logic (realistic relationships)
risk_score = (
    -0.00002 * income
    + 0.00005 * debt
    - 0.005 * credit_score
    + 0.00004 * loan_amount
    - 0.02 * employment_years
    + np.random.normal(0, 0.5, n)
)

default = (risk_score > np.percentile(risk_score, 60)).astype(int)

df = pd.DataFrame({
    "income": income,
    "debt": debt,
    "credit_score": credit_score,
    "loan_amount": loan_amount,
    "employment_years": employment_years,
    "age": age,
    "default": default
})

df.head()

,income,debt,credit_score,loan_amount,employment_years,age,default
0,67450.712295,33993.554366,602.737521,2106.309305,7.275270,28.429809,0
1,57926.035482,29246.336829,639.883693,14675.379871,1.281045,41.269603,1
2,69715.328072,20596.303699,594.530606,20036.733599,14.586921,34.511861,0
3,82845.447846,13530.632223,628.442693,47652.251888,7.770156,22.547007,1
4,56487.699379,26982.233136,517.446973,31678.637494,9.212794,35.218390,1


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("default", axis=1).values
y = df["default"].values.reshape(-1, 1)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [8]:
class CreditNet(nn.Module):
    def __init__(self):
        super(CreditNet, self).__init__()
        self.fc1 = nn.Linear(6, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

model = CreditNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [9]:
epochs = 200

for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.7128
Epoch 20, Loss: 0.5135
Epoch 40, Loss: 0.3445
Epoch 60, Loss: 0.3275
Epoch 80, Loss: 0.3177
Epoch 100, Loss: 0.3118
Epoch 120, Loss: 0.3064
Epoch 140, Loss: 0.2992
Epoch 160, Loss: 0.2894
Epoch 180, Loss: 0.2766


In [10]:
with torch.no_grad():
    predictions = model(X_test)
    predicted = (predictions > 0.5).float()
    accuracy = (predicted == y_test).float().mean()

print("Test Accuracy:", accuracy.item())

Test Accuracy: 0.8100000023841858
